In [1]:
import torch
import os
import gc
import json
import re
import subprocess
import pandas as pd
from torch import cuda
from openai import OpenAI
from IPython.display import Markdown, display
from google.colab import files, drive

# USER INPUT
user_base_url="https://openrouter.ai/api/v1"
user_api_key="api_key"

user_model = "anthropic/claude-3.7-sonnet"
user_filename_id = "Claude3_conceptual"
user_max_token = 16384

print("Done")

Done


In [2]:
class LLMSolver:
  def __init__(self, ulr=None, api=None, model=None, filename_id=None, token=0, csv_file_path=None):
    """Initialize OpenAI client with environment variable"""
    self.client=OpenAI(
        base_url=ulr,
        api_key=api,
    )
    self.model = model
    self.filename_id = filename_id
    self.max_token = token
    self.csv_file_path = csv_file_path

  def setup_drive(self):
    """Mount Google Drive and create output folder"""
    try:
      drive.mount('/content/drive')
      csv_file_path = self.csv_file_path
      os.makedirs(csv_file_path, exist_ok=True)
      print(f"Conceptual responses will be saved to: {csv_file_path}")
    except Exception as e:
      print(f"Error setting up drive: {e}")

  def load_data(self):
    """Upload and load CSV file"""
    try:
      uploaded = files.upload()
      filename = list(uploaded.keys())[0]
      df = pd.read_csv(filename)
      df = df.dropna(how='all')
      display(df.iloc[:20,:8])
      total_rows = len(df)/5
      print(f"Total rows in dataset: {total_rows}")

      return df, total_rows

    except Exception as e:
      print(f"Error loading data: {e}")
      return None

  def prompt_design(self, mode):
    """

    NSD = Non-specific domain promt
    IAQ = Indoor Air Quality specific domain promt

    """
    PROMPT_GSR = """
    Structure your responses in one paragraph, maximum 200 words
    """

    mode_prompts = {
        "NSD": "You will be provided with a conceptual problem. Please solve it using the instructions above.",
        "IAQ": "You are an expert in the Indoor Air Quality Engineering field. You will be provided with a calculation problem. Please solve it using the instructions above."
    }

    return PROMPT_GSR + mode_prompts.get(mode, mode_prompts[mode])

  def ask_model(self, prompt, mode=None):
    """
    Generate response from the AI model
    """
    try:
      full_prompt = self.prompt_design(mode) + prompt

      completion = self.client.chat.completions.create(
        extra_headers={
            "HTTP-Referer": "UIUC",
            "X-Title": "Benchmarking LLM IAQ",
        },
        model=self.model,
        messages=[{"role": "user", "content": full_prompt}],
        max_tokens=self.max_token
      )

      return completion.choices[0].message.content

    except Exception as e:
      print(f"Error in ask_model: {e}")
      return None

  def display_Markdown(self,response):
    """Display the answers in Colab interface"""
    display(Markdown(response))

  def update_csv_with_responses(self, response, row_index, csv_path):
    """Save model response to CSV file"""
    try:
        # Load the existing CSV
        df = pd.read_csv(csv_path)

        # Add model responses to column "Model Answer" in the CSV file
        df["Model Answer"] = df["Model Answer"].astype('object')
        df.loc[row_index, "Model Answer"] = str(response)

        # Save back to the same file
        df.to_csv(csv_path, index=False)
        print(f"Response saved for row {row_index}")

        return True
    except Exception as e:
        print(f"Error updating CSV: {e}")
        return None

  def process_problems(self, df, batch_size, answer_mode, max_problems, displays_Colab):
    """Process problems from DataFrame"""
    try:
        for i_batch in range(1, batch_size + 1):
            for index, row in df.iloc[0:5*max_problems:5].iterrows():
                example_id = str(df.iloc[index+i_batch-1]["Combined_No"]).strip()
                problem = str(df.iloc[index+i_batch-1]["Question"]).strip()

                print(f"Generating response for {example_id} in mode {answer_mode} of batch: {i_batch}")

                # Generate response
                response = self.ask_model(problem, answer_mode)
                if not response:
                    print(f"Failed to generate save_status")
                    continue

                if displays_Colab == True:
                  # Display result
                  self.display_Markdown(response)

                # Save to CSV file
                save_status = self.update_csv_with_responses(response, index+i_batch-1, self.csv_file_path)
                if save_status:
                    print(f"Successfully save response for {example_id} in mode {answer_mode} of batch: {i_batch}")
                print(20*"-")
        print("Done processing all problems")

        return True

    except Exception as e:
        print(f"Error in process_problems: {e}")

        return False

# Notice
print("Class and Function declaration > Done")

Class and Function declaration > Done


In [4]:
# Initialize the solver
try:
  solver = LLMSolver(
          user_base_url,
          user_api_key,
          user_model,
          user_filename_id,
          user_max_token,
          None
  )
  print("Successfully initializing the LLMSolver")
except Exception as e:
  print(f"Error in initializing the solver: {e}")

# Setup Google Drive and output file path
try:
  solver.setup_drive()
  print("Successfully initializing the Google Driver")
except Exception as e:
  print(f"Error in initializing the Google Driver: {e}")

# Load CSV file
try:
  df, total_rows = solver.load_data()
  print("Successfully loading CSV file")
except Exception as e:
  print(f"Error in loading CSV file: {e}")


Successfully initializing the LLMSolver
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Error setting up drive: expected str, bytes or os.PathLike object, not NoneType
Successfully initializing the Google Driver


Saving Discussion_Dataset_Cleaned_Answer_CSV_oldanswer.csv to Discussion_Dataset_Cleaned_Answer_CSV_oldanswer.csv


,No.,Sub_No.,Combined_No,Chapter,Question,Trials,Reference Answer,Model Answer
0,1.0,1.0,1_1,Chapter 1,What constitutes an air contaminant?,1.0,Air contaminants are generally classified into...,NaN
1,1.0,2.0,1_2,Chapter 1,What constitutes an air contaminant?,2.0,Air contaminants are generally classified into...,NaN
2,1.0,3.0,1_3,Chapter 1,What constitutes an air contaminant?,3.0,Air contaminants are generally classified into...,NaN
3,1.0,4.0,1_4,Chapter 1,What constitutes an air contaminant?,4.0,Air contaminants are generally classified into...,NaN
4,1.0,5.0,1_5,Chapter 1,What constitutes an air contaminant?,5.0,Air contaminants are generally classified into...,NaN
5,2.1,1.0,2.1_1,Chapter 1,What is the definition of indoor air quality?,1.0,Indoor air quality refers to the degree of pol...,NaN
6,2.1,2.0,2.1_2,Chapter 1,What is the definition of indoor air quality?,2.0,Indoor air quality refers to the degree of pol...,NaN
7,2.1,3.0,2.1_3,Chapter 1,What is the definition of indoor air quality?,3.0,Indoor air quality refers to the degree of pol...,NaN
8,2.1,4.0,2.1_4,Chapter 1,What is the definition of indoor air quality?,4.0,Indoor air quality refers to the degree of pol...,NaN
9,2.1,5.0,2.1_5,Chapter 1,What is the definition of indoor air quality?,5.0,Indoor air quality refers to the degree of pol...,NaN


Total rows in dataset: 174.0
Successfully loading CSV file


In [ ]:
# Please specify the CSV file path saved on Google Driver

solver.csv_file_path = "/content/Discussion_Dataset_Cleaned_Answer_CSV_oldanswer.csv"

# Run model
# Model will automatically save answers to CSV file
run_status = solver.process_problems(
    df=df,
    batch_size = 5,
    answer_mode = "NSD",
    max_problems=174,
    displays_Colab = True,
)

if run_status:
  print(100*"*")
  print("Successfully generate response and save to CSV.")

Generating response for 1_1 in mode NSD of batch: 1


An air contaminant refers to any substance that pollutes the atmosphere and potentially poses risks to human health, wildlife, or the environment. These substances can be particulate matter (such as dust, smoke, or soot), gases (like carbon monoxide, sulfur dioxide, or volatile organic compounds), biological agents (including mold spores, bacteria, or viruses), or radioactive materials. Air contaminants can originate from both natural sources (such as volcanic eruptions, forest fires, or pollen) and anthropogenic activities (including industrial processes, vehicle emissions, or waste incineration). To be classified as a contaminant, these substances typically exceed natural background levels or have the potential to cause adverse effects. Regulatory agencies like the Environmental Protection Agency establish threshold concentrations for many contaminants, above which they are considered harmful. The classification can also depend on context - substances that are normally present in air may become contaminants when their concentration increases significantly or when they appear in environments where they wouldn't naturally occur.

Response saved for row 0
Successfully save response for 1_1 in mode NSD of batch: 1
--------------------
Generating response for 2.1_1 in mode NSD of batch: 1


Indoor air quality (IAQ) refers to the condition of air within enclosed spaces, particularly as it relates to the health, comfort, and well-being of building occupants. It encompasses the concentration of pollutants, temperature, humidity, ventilation rates, and other environmental factors that can affect human health and comfort. Common IAQ concerns include the presence of particulate matter, volatile organic compounds (VOCs), carbon monoxide, radon, mold, bacteria, and allergens. Poor IAQ can lead to various health issues ranging from minor irritations like headaches and fatigue to serious conditions such as respiratory diseases, heart disease, and cancer. Factors affecting IAQ include building materials, furnishings, cleaning products, outdoor pollution infiltration, HVAC systems, human activities, and biological contaminants. Maintaining good IAQ typically involves proper ventilation, source control of pollutants, air cleaning, and regular maintenance of building systems.

Response saved for row 5
Successfully save response for 2.1_1 in mode NSD of batch: 1
--------------------
Generating response for 3_1 in mode NSD of batch: 1


...

Response saved for row 437
Successfully save response for 9_3 in mode NSD of batch: 3
--------------------
Generating response for 10_3 in mode NSD of batch: 3


When designing a sampler head in calm-air sampling, we encounter a critical particle size where the maximum allowable diameter equals the minimum required diameter. For particles larger than this threshold, the calculated minimum diameter exceeds the maximum, creating an impossible design constraint. In this circumstance, the recommended approach is to acknowledge that perfect sampling efficiency cannot be achieved for these larger particles and to establish performance criteria that prioritize the most important particle size range for the specific application. Engineers should either: (1) optimize the design for the particle size range of greatest interest, accepting reduced efficiency for larger particles; (2) implement a pre-separator or impactor stage to remove particles above the critical size before they enter the main sampling system; or (3) characterize the sampling bias for larger particles and apply appropriate correction factors during data analysis. Documentation of these limitations and the resulting sampling efficiency curve across all particle sizes is essential for proper interpretation of sampling results.

Response saved for row 442
Successfully save response for 10_3 in mode NSD of batch: 3
--------------------
Generating response for 11.1_3 in mode NSD of batch: 3


No, the absence of a sampling efficiency equation for calm-air sampling doesn't mean it lacks efficiency concerns. While calm-air environments eliminate wind-related biases (such as those addressed in equations for moving-air sampling), other factors still affect collection efficiency, including gravitational settling, particle bounce, inlet design, electrostatic effects, and thermophoresis. Particles may settle before being collected, bounce off collection surfaces, be lost to walls due to poor inlet design, be attracted or repelled by static charges, or move in response to temperature gradients. Additionally, calm-air conditions are rare in practice; most environments have some air movement that affects sampling. Therefore, even without a specific equation, researchers must still consider these physical phenomena when designing sampling protocols and interpreting results from calm-air environments, making efficiency a relevant concern regardless of air movement conditions.

Response saved for row 447
Successfully save response for 11.1_3 in mode NSD of batch: 3
--------------------
Generating response for 11.2_3 in mode NSD of batch: 3
